# Controlling Context

As I continue to add functionalities, I am finding that:
1) The qualitiy of the output seems to be hitting a point where the agents aren't really improving.
2) Just pasting the promprt into the web interface of an LLM provider (OpenAI, Claude, etc) yields similar results.  I feel I am hitting the point where the focus needs to shift to getting the agents to do something that isn't possible with out using agentic AI.

I think I can improve on things with the following:
- Looking at the traces (https://platform.openai.com/traces), I see that the entire context gets passed from agent to agent.  I am really only interested in having the agents create, update and refine the trip plan.  Maybe just passsing the current trip plan from agent to agent will increase efficiency and give a greater degree of control over what the agents are doing.
- A more structed format would help for consistentcy.  Maybe having the agents maintain a JSON formatted travel plan would help the agents to focus on the data, then as a final step, convert that data into a formatted output.  
- I am also interested in seeing how good an agent can be at determining how to use a set of tools to generate the final output.  I want to define a set of agenets/tools to see how well an agent can be at applying a set of tools to accomplish a task.  Can it be creative at usign a set of tools?


In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from controlling_context_instructions import TripPlannerInstructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


### The evaluation and wrire_file agent instructions

The evaluation agent remains the same as the previous notebook.

I had to go through a couple if iterations of the instructions to give to the handoff agent that is responsibible for writing the file to disk.  It needs
Very specific instructions to save the correct content to disk.

In [2]:
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from dataclasses import dataclass

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
output_file=os.path.join(sandbox_path, "10_trip_plan_controlling_context.json")
# Generate custom instructions for the trip planner agent

# define parameters for different MCP server implementations
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
serper_params = {"command": "uvx", "args": ["serper-mcp-server"], "env": {"SERPER_API_KEY": os.environ.get('SERPER_API_KEY')}}

@dataclass
class TripPlannerContext:
    """
    Context object passed between agents during trip planning.
    
    This dataclass maintains the state and metadata for a trip planning session,
    allowing agents to share and update information as the plan evolves.
    """

    trip_details_json_path: str
    """Absolute file path where the structured trip plan JSON should be saved.
    Agents responsible for persisting data should write to this location."""

    requred_action: str
    """The specific action the agent is required to perform in this step.
    This guides the agent's focus and ensures alignment with the overall plan."""

## Multiple agents passing around a specific context.




In [ ]:
from agents import ModelSettings


planner = TripPlannerInstructions(
    output_file=output_file
)

manager_agent_instructions = planner.get_instructions_for_manager()
activities_expert_agent_instructions = planner.get_instructions_for_activities_expert()
accommodations_expert_agent_instructions = planner.get_instructions_for_accommodations_expert()
transportation_expert_agent_instructions = planner.get_instructions_for_transportation_expert()  
trip_evaluation_expert_agent_instructions = planner.get_instructions_for_trip_evaluation_expert()

print(manager_agent_instructions)
print("*" * 120)

# Set the base_url to your local Ollama instance
# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"
GROQ_URL = "https://api.groq.com/openai/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

local_client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY
)

# Specify the model you pulled with Ollama
OPENAI_OLLAMA_MODEL_NAME = "gpt-oss:20b" 
OLLAMA_MODEL_NAME = "gpt-oss_131k_context:20b" 


model_local_oss = OpenAIChatCompletionsModel(
    openai_client=local_client,
    model=OLLAMA_MODEL_NAME
)

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=45) as mcp_server_serper:

        # Define the transportation evaluation expert agent
        transportation_expert_agent = Agent[TripPlannerContext](
            model=model_local_oss,
            name="Transportation_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=transportation_expert_agent_instructions
        )

        #Define the activities expert agent
        activities_expert_agent = Agent[TripPlannerContext](
            model=model_local_oss,
            name="Activities_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=activities_expert_agent_instructions
        )

        #Define the accommodations expert agent
        accommodations_expert_agent = Agent[TripPlannerContext](
            model=model_local_oss,
            name="Accommodations Expert Agent",
            mcp_servers=[mcp_server_serper],
            instructions=accommodations_expert_agent_instructions
        )

        #Define the trip evaluation expert agent
        trip_evaluation_expert_agent = Agent[TripPlannerContext](
            model=model_local_oss,
            name="Trip_Evaluation_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=trip_evaluation_expert_agent_instructions
        )


        # Define the trip planner agent with the transportation evaluation expert as a tool
        trip_planner_manager_agent = Agent[TripPlannerContext](
            model=model_local_oss,
            name="Trip Planner Agent",
            instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
            mcp_servers=[mcp_server_files],
            tools=[
                transportation_expert_agent.as_tool(
                    tool_name="Transportation_Expert_Agent",
                    tool_description="An expert agent that provides transportation options and advice for trip planning.",    
                ),
                activities_expert_agent.as_tool(
                    tool_name="Activities_Expert_Agent",
                    tool_description="An expert agent that provides recommendations and advice on activities for trip planning.",    
                ),
                accommodations_expert_agent.as_tool(
                    tool_name="Accommodations_Expert_Agent",
                    tool_description="An expert agent that provides recommendations and advice on accommodations for trip planning.",    
                ),
                trip_evaluation_expert_agent.as_tool(
                    tool_name="Trip_Evaluation_Expert_Agent",
                    tool_description="An expert agent that evaluates the trip plan for clarity and completeness.",    
                ),
            ],
            model_settings=ModelSettings(tool_choice="required")
        )
        with trace("Trip Planner Agent Ollama"):
            result = await Runner.run(trip_planner_manager_agent, manager_agent_instructions, max_turns=100)
            print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

You will have several tools at your disposal to help you complete this task.  Use the tools repeatedly as needed to gather information and refine the itinerary. The tools available to you are:
1. Activities_Expert_Agent: Provide this tool with a Location and a date and the tool will return a list of recommended activities and attractions for that location on that date.
2. Transportation_Expert_Agent: Use this tool to get detailed transportation options between locations, including specific train/subway/bus lines, departure times, durations, and costs.
3. Trip_Evaluation_Expert_Agent: As you refine the trip plan, this tool will evaluate the plan and provide feedback on the clarity and completeness of the plan.
4. Accommodations_Expert_Agent: Provide this tool with a Location and date ra

## All available GROQ models

The code below will list all models available from GROQ

In [ ]:
# list all groq models
import requests
import json
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)
print(json.dumps(response.json(), indent=4))

{
    "object": "list",
    "data": [
        {
            "id": "openai/gpt-oss-20b",
            "object": "model",
            "created": 1754407957,
            "owned_by": "OpenAI",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 65536
        },
        {
            "id": "moonshotai/kimi-k2-instruct-0905",
            "object": "model",
            "created": 1757046093,
            "owned_by": "Moonshot AI",
            "active": true,
            "context_window": 262144,
            "public_apps": null,
            "max_completion_tokens": 16384
        },
        {
            "id": "meta-llama/llama-4-scout-17b-16e-instruct",
            "object": "model",
            "created": 1743874824,
            "owned_by": "Meta",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 8192
        },
        {
    